In [ ]:
pip install -U langchain langchain_core langchain-community

In [ ]:
pip install getpass4

In [ ]:
pip install crewai

In [ ]:
%pip install --upgrade --quiet huggingface_hub

In [ ]:
pip install langchain_huggingface

In [ ]:
import os
from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = getpass()
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

In [ ]:
import sqlite3

# Create the SQLite database and table
conn = sqlite3.connect('dummy_data.db')
cursor = conn.cursor()

# Drop the sales table if it already exists
cursor.execute('DROP TABLE IF EXISTS sales')

# Create the sales table
cursor.execute('''
    CREATE TABLE sales (
        sl_no INTEGER PRIMARY KEY,
        product TEXT,
        shipping_location TEXT,
        shipping_costs INTEGER,
        actual_costs INTEGER,
        selling_price INTEGER,
        selling_location TEXT,
        profits INTEGER
    )
''')

# Insert dummy data into the sales table
data = [
    (1, 'Flask', 'Bangalore', 50, 200, 300, 'Mumbai', 100),
    (2, 'Flask', 'Delhi', 40, 200, 310, 'Pune', 110),
    (3, 'Flask', 'Chennai', 60, 200, 320, 'Hyderabad', 120),
    (6, 'Bottle', 'Bangalore', 30, 100, 200, 'Mumbai', 100),
    (7, 'Bottle', 'Delhi', 25, 100, 210, 'Pune', 110),
    (8, 'Bottle', 'Chennai', 35, 100, 220, 'Hyderabad', 120),
    (11, 'Mug', 'Bangalore', 20, 50, 100, 'Mumbai', 50),
    (12, 'Mug', 'Delhi', 18, 50, 110, 'Pune', 60),
    (13, 'Mug', 'Chennai', 22, 50, 120, 'Hyderabad', 70),
    (16, 'Cup', 'Bangalore', 15, 30, 70, 'Mumbai', 40),
    (17, 'Cup', 'Delhi', 12, 30, 75, 'Pune', 45),
    (18, 'Cup', 'Chennai', 18, 30, 80, 'Hyderabad', 50),
    (21, 'Plate', 'Bangalore', 25, 60, 120, 'Mumbai', 60),
    (22, 'Plate', 'Delhi', 22, 60, 125, 'Pune', 65),
    (23, 'Plate', 'Chennai', 28, 60, 130, 'Hyderabad', 70),
    (26, 'Pan', 'Bangalore', 35, 150, 250, 'Mumbai', 100),
    (27, 'Pan', 'Delhi', 32, 150, 260, 'Pune', 110),
    (28, 'Pan', 'Chennai', 38, 150, 270, 'Hyderabad', 120),
    (31, 'Pot', 'Bangalore', 25, 80, 160, 'Mumbai', 80),
    (32, 'Pot', 'Delhi', 22, 80, 165, 'Pune', 85),
    (33, 'Pot', 'Chennai', 28, 80, 170, 'Hyderabad', 90),
    (36, 'Bowl', 'Bangalore', 10, 20, 50, 'Mumbai', 30),
    (37, 'Bowl', 'Delhi', 8, 20, 55, 'Pune', 35),
    (38, 'Bowl', 'Chennai', 12, 20, 60, 'Hyderabad', 40),
    (41, 'Spoon', 'Bangalore', 5, 10, 30, 'Mumbai', 20),
    (42, 'Spoon', 'Delhi', 4, 10, 32, 'Pune', 22),
    (43, 'Spoon', 'Chennai', 6, 10, 34, 'Hyderabad', 24),
    (46, 'Fork', 'Bangalore', 5, 15, 35, 'Mumbai', 20),
    (47, 'Fork', 'Delhi', 4, 15, 38, 'Pune', 23),
    (48, 'Fork', 'Chennai', 6, 15, 41, 'Hyderabad', 26)
]

cursor.executemany('''
    INSERT INTO sales (sl_no, product, shipping_location, shipping_costs, actual_costs, selling_price, selling_location, profits)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
''', data)

conn.commit()
conn.close()

In [ ]:
from langchain.agents.agent_toolkits import SQLDatabaseToolkit 
from langchain.sql_database import SQLDatabase 

db = SQLDatabase.from_uri("sqlite:///dummy_data.db")
print("Database dialect:", db.dialect)
print("Usable table names:", db.get_usable_table_names())

# Run a test query
test_query_result = db.run("SELECT * FROM sales LIMIT 10;")
print("Test query result:", test_query_result)
print(db.get_table_info())

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint

model_id = "openai-community/gpt2-large"

llm = HuggingFaceEndpoint(
    repo_id=model_id,
    max_new_tokens=250,
    top_k=15,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.2,
    # repetition_penalty=1.03,
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)

# print(llm.invoke("What is Deep Learning?"))

In [ ]:
from crewai import Agent
from textwrap import dedent

class CustomAgents:
    def __init__(self):
        self.huggingFace = llm

    def sql_agent(self):
        return Agent(
            role="Text-to-SQL Agent",
            backstory=dedent(f"""I'm an expert in converting natural language to SQL queries and executing them on a database."""),
            goal=dedent(f"""Convert user text to efficient SQL code, execute it on the database and return the output in a concise manner"""),
            # tools=[tool_1, tool_2],
            allow_delegation=False,
            verbose=True,
            llm=self.huggingFace,
        )

In [ ]:
from crewai import Task
from textwrap import dedent

class CustomTasks:
    def __tip_section(self):
        return "Do your BEST WORK, you get a $10,000 commission!"

    def conversion(self, agent, database, tables, dialect, question):
        return Task(
            description=dedent(
                f"""
            Convert the {question} to SQL using the data from the mentioned database and the available tables.
            This is the actual database {database} and schema along with example of data from the database {tables}.

            This is an example query {"SELECT * FROM sales LIMIT 10;"}
            Make sure to use the correct format and syntax based on the database type {dialect}
                """
            ),
            expected_output="Readable text summarizing the query results",
            agent=agent
        )

In [ ]:
import os
from crewai import Agent, Task, Crew, Process

from textwrap import dedent

class CustomCrew:
    def __init__(self, database, tables, dialect, question):
        self.database = database
        self.tables = tables
        self.dialect = dialect
        self.question = question

    def run(self):
        # Define your custom agents and tasks in agents.py and tasks.py
        agents = CustomAgents()
        tasks = CustomTasks()

        # Define your custom agents and tasks here
        custom_agent_1 = agents.sql_agent()

        # Custom tasks include agent name and variables as input
        custom_task_1 = tasks.conversion(
            custom_agent_1,
            self.database,
            self.tables,
            self.dialect,
            self.question
        )

        # Define your custom crew here
        crew = Crew(
            agents=[custom_agent_1],
            tasks=[custom_task_1],
            verbose=True,
        )

        result = crew.kickoff()
        return result


# This is the main function that you will use to run your custom crew.
if __name__ == "__main__":
    print("## Welcome to Crew AI Template")
    print("-------------------------------")
    database = db
    tables = db.get_table_info()
    dialect = db.dialect
    question = input("What would you like to know?")

    custom_crew = CustomCrew(database, tables, dialect, question)
    result = custom_crew.run()
    print("\n\n########################")
    print("## Here is your custom crew run result:")
    print("########################\n")
    print(result)